In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import sklearn
import pandas as pd
try:
    import seaborn as sns
except ImportError:
    sns = None


In [2]:
from pathlib import Path
DATA_ROOT = Path('/home/pengfei/Downloads/data')
DATA_ROOT


PosixPath('/home/pengfei/Downloads/data')

# Data folder column/body-part audit

这部分是给 `/home/pengfei/Downloads/data` 下面每个数据集做的结构说明：包含什么数据、列/字段代表什么、对应哪个身体部位、单位/频率，以及哪些地方是从本地文件明确读到的，哪些地方需要继续复核。

注意：有些数据集的列不是普通 CSV column，而是 tensor 维度、JSON key、或每行固定顺序拼接的 body part。下面会分别说明。


In [1]:
from pathlib import Path
import csv
import json
import re
import zipfile

import numpy as np
import pandas as pd
import torch

DATA_ROOT = Path('/home/pengfei/Downloads/data')
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
DATA_ROOT


PosixPath('/home/pengfei/Downloads/data')

## 1. Dataset overview

`raw files / columns` 是原始或局部原始数据的结构；`body parts` 是我能从本地 metadata、README、header、或已有 converter 代码确认/推断的部位映射。


In [4]:
dataset_overview = [
    {
        'dataset': 'CeTI-Age-Kinematics',
        'local_root': 'data/CeTI-Age-Kinematics/ceti-age-kinematics',
        'files': '*_motion.tsv, *_channels.tsv, *_motion.json, participants.tsv',
        'what_data': 'Rokoko/suit solved full-body kinematics: segment/joint positions and joint angles over time.',
        'columns_or_fields': 'Timestamp; <BodyPart>_position_x/y/z; joint-angle columns such as LeftHip_flexion.',
        'body_parts': 'Pelvis, upper/lower legs, feet, chest/thorax, shoulders, upper arms, forearms, hands, head/neck, etc. Exact list is read from *_channels.tsv below.',
        'units_fps': 'Timestamp ms; positions cm; angles deg; commonly 100 Hz from metadata/header.',
        'notes': 'No raw IMU channels found in the files used here; in unified processing IMU is zero-filled and this is motion_only.',
    },
    {
        'dataset': 'MobilePoser',
        'local_root': 'data/mobileposer',
        'files': '*.pt, eval/*.pt',
        'what_data': 'Processed sparse-IMU + SMPL body data stored as PyTorch tensors/lists.',
        'columns_or_fields': 'keys: joint, pose, shape, tran, acc, ori, contact; dimensions map to time, sensor/joint/body parameter axes.',
        'body_parts': '6 IMUs: Pelvis, LeftLowerLeg, RightLowerLeg, Head, LeftForeArm, RightForeArm. joint/pose use SMPL 24-joint order.',
        'units_fps': 'joint/tran treated as meters; ori/pose are rotations; acc is processed acceleration; usually 60 Hz.',
        'notes': 'TotalCapture, DIP, IMUPoser are eval-only in our repack.',
    },
    {
        'dataset': 'VIDIMU',
        'local_root': 'data/vidimu',
        'files': 'dataset/videoandimus/**/*.csv, *.raw, *.sto, *_Npose.csv',
        'what_data': 'Video/IK 3D keypoints plus IMU orientation files.',
        'columns_or_fields': 'CSV has <keypoint>_x/y/z. STO has time plus quaternion columns like pelvis_imu, femur_r_imu.',
        'body_parts': 'CSV keypoints include pelvis, hips, knees, ankles, toes, heels, head/face landmarks, shoulders, elbows, wrists, hand landmarks. STO IMUs include pelvis, right/left femur, right/left tibia in sample files.',
        'units_fps': 'CSV coordinates are mm-scale in our converter and converted by /1000; STO DataRate=50 and DataType=Quaternion.',
        'notes': 'Current unified converter uses CSV joints and optional .raw orientation; no acceleration is populated.',
    },
    {
        'dataset': 'ULTRA-MoCap',
        'local_root': 'data/ULTRA-MoCap',
        'files': 'Raw Marker/*.trc, IMU/*.csv, EMG/*.csv, Processed IK/*.csv',
        'what_data': 'Vicon marker trajectories, IMU acc/gyro, EMG, and OpenSim/IK joint coordinates.',
        'columns_or_fields': 'TRC has marker x/y/z triples. IMU CSV has ACCXn/ACCYn/ACCZn and GYROXn/GYROYn/GYROZn. IK CSV has time plus joint coordinates/angles.',
        'body_parts': 'Marker names map to anatomical landmarks; current converter maps LASI/RASI/LPSI/RPSI to pelvis, limb markers to legs/feet/toes. IMU sensor IDs are mapped in current code to Pelvis, T8, LeftUpperArm, RightUpperArm, LeftForeArm, RightForeArm, but this placement should be verified against ULTRA docs.',
        'units_fps': 'TRC marker positions are mm and converted by /1000; TRC rate usually 100 Hz. IMU ACC scaling in current code is /1000 but marked uncertain.',
        'notes': 'Orientation is not directly available from the IMU CSV reader; converter uses identity orientation placeholder and stores gyro separately.',
    },
    {
        'dataset': 'KeraalDataset',
        'local_root': 'data/KeraalDataset',
        'files': 'vicon, kinect, openpose JSON, blazepose JSON, annotations .anvil, videos',
        'what_data': 'Multi-modal exercise recordings for patient/healthy groups: Vicon/Kinect skeletons, 2D/3D pose estimates, annotations, videos.',
        'columns_or_fields': 'Vicon rows concatenate 17 targets * (x,y,z,qx,qy,qz,qw). Kinect rows concatenate 25 joints * (x,y,z,qx,qy,qz,qw). OpenPose/BlazePose JSON is frame -> joint -> coordinates.',
        'body_parts': 'Vicon 17 target order and Kinect 25 joint order are listed below from the local README. OpenPose COCO and BlazePose 33 keypoints are also listed below.',
        'units_fps': 'Kinect position likely meters; Vicon position scale not explicitly in README here and should be checked from original files/docs; BlazePose z is relative to video plane.',
        'notes': 'No synchronized sparse IMU stream compatible with current full-body IMU training schema.',
    },
    {
        'dataset': 'rehab_data / 12112951',
        'local_root': 'data/rehab_data/12112951',
        'files': 'dataset.zip, metadata.txt',
        'what_data': 'Delsys Trigno Avanti EMG + IMU for rehab exercises, plus subject metadata and labels.',
        'columns_or_fields': '8 sensors; each sensor has 1 EMG channel and 6 IMU channels: acc x/y/z, gyro x/y/z.',
        'body_parts': 'Sensors on right/left rectus femoris, hamstrings, tibialis anterior, gastrocnemius. Exact placement table is read below from metadata.txt.',
        'units_fps': 'EMG mV at 1259.259 Hz; accelerometer G; gyroscope deg/s; IMU 148.148 Hz.',
        'notes': 'Good for rehab exercise classification or EMG/IMU modeling, but not full-body joint-position GT.',
    },
]

overview_df = pd.DataFrame(dataset_overview)
overview_df


,dataset,local_root,files,what_data,columns_or_fields,body_parts,units_fps,notes
0,CeTI-Age-Kinematics,data/CeTI-Age-Kinematics/ceti-age-kinematics,"*_motion.tsv, *_channels.tsv, *_motion.json, p...",Rokoko/suit solved full-body kinematics: segme...,Timestamp; <BodyPart>_position_x/y/z; joint-an...,"Pelvis, upper/lower legs, feet, chest/thorax, ...",Timestamp ms; positions cm; angles deg; common...,No raw IMU channels found in the files used he...
1,MobilePoser,data/mobileposer,"*.pt, eval/*.pt",Processed sparse-IMU + SMPL body data stored a...,"keys: joint, pose, shape, tran, acc, ori, cont...","6 IMUs: Pelvis, LeftLowerLeg, RightLowerLeg, H...",joint/tran treated as meters; ori/pose are rot...,"TotalCapture, DIP, IMUPoser are eval-only in o..."
2,VIDIMU,data/vidimu,"dataset/videoandimus/**/*.csv, *.raw, *.sto, *...",Video/IK 3D keypoints plus IMU orientation files.,CSV has <keypoint>_x/y/z. STO has time plus qu...,"CSV keypoints include pelvis, hips, knees, ank...",CSV coordinates are mm-scale in our converter ...,Current unified converter uses CSV joints and ...
3,ULTRA-MoCap,data/ULTRA-MoCap,"Raw Marker/*.trc, IMU/*.csv, EMG/*.csv, Proces...","Vicon marker trajectories, IMU acc/gyro, EMG, ...",TRC has marker x/y/z triples. IMU CSV has ACCX...,Marker names map to anatomical landmarks; curr...,TRC marker positions are mm and converted by /...,Orientation is not directly available from the...
4,KeraalDataset,data/KeraalDataset,"vicon, kinect, openpose JSON, blazepose JSON, ...",Multi-modal exercise recordings for patient/he...,"Vicon rows concatenate 17 targets * (x,y,z,qx,...",Vicon 17 target order and Kinect 25 joint orde...,Kinect position likely meters; Vicon position ...,No synchronized sparse IMU stream compatible w...
5,rehab_data / 12112951,data/rehab_data/12112951,"dataset.zip, metadata.txt",Delsys Trigno Avanti EMG + IMU for rehab exerc...,8 sensors; each sensor has 1 EMG channel and 6...,"Sensors on right/left rectus femoris, hamstrin...",EMG mV at 1259.259 Hz; accelerometer G; gyrosc...,Good for rehab exercise classification or EMG/...


## 2. CeTI: columns, units, body parts

CeTI 的 `*_channels.tsv` 是最可靠的列说明来源；下面直接读它，输出每个 motion column 的 component、tracked point、type、unit。


In [5]:
ceti_motion = DATA_ROOT / 'CeTI-Age-Kinematics/ceti-age-kinematics/sub-d02/ses-01/motion/sub-d02_ses-01_task-c01_tracksys-rokokosmartsuit1_run-01_motion.tsv'
ceti_channels = DATA_ROOT / 'CeTI-Age-Kinematics/ceti-age-kinematics/sub-d02/ses-01/motion/sub-d02_ses-01_task-c01_tracksys-rokokosmartsuit1_run-01_channels.tsv'

ceti_channel_df = pd.read_csv(ceti_channels, sep='\t')
print('motion file:', ceti_motion)
print('n motion columns:', len(pd.read_csv(ceti_motion, sep='\t', nrows=0).columns))
print('n channel metadata rows:', len(ceti_channel_df))
ceti_channel_df.head(120)


motion file: /home/pengfei/Downloads/data/CeTI-Age-Kinematics/ceti-age-kinematics/sub-d02/ses-01/motion/sub-d02_ses-01_task-c01_tracksys-rokokosmartsuit1_run-01_motion.tsv
n motion columns: 103
n channel metadata rows: 103


,name,component,tracked_point,type,units
0,Timestamp,NaN,NaN,MISC,ms
1,Pelvis_position_x,x,Pelvis,POS,cm
2,Pelvis_position_y,y,Pelvis,POS,cm
3,Pelvis_position_z,z,Pelvis,POS,cm
4,Pelvis_extension,NaN,Pelvis,ORNT,deg
5,Pelvis_lateral_flexion_rotation,NaN,Pelvis,ORNT,deg
6,Pelvis_axial_rotation,NaN,Pelvis,ORNT,deg
7,LeftUpperLeg_position_x,x,LeftUpperLeg,POS,cm
8,LeftUpperLeg_position_y,y,LeftUpperLeg,POS,cm
9,LeftUpperLeg_position_z,z,LeftUpperLeg,POS,cm


In [6]:
def summarize_ceti_columns(channels_df):
    rows = []
    for _, r in channels_df.iterrows():
        name = str(r.get('name', ''))
        if name == 'Timestamp':
            rows.append({'column': name, 'body_part': 'global time', 'signal': 'timestamp', 'axis_or_component': 'time', 'unit': r.get('units', 'ms')})
            continue
        m = re.match(r'(.+)_position_([xyz])$', name)
        if m:
            rows.append({'column': name, 'body_part': m.group(1), 'signal': '3D position', 'axis_or_component': m.group(2), 'unit': r.get('units', 'cm')})
        else:
            rows.append({'column': name, 'body_part': r.get('tracked_point', ''), 'signal': r.get('type', 'joint angle / rotation'), 'axis_or_component': r.get('component', ''), 'unit': r.get('units', 'deg')})
    return pd.DataFrame(rows)

ceti_column_meanings = summarize_ceti_columns(ceti_channel_df)
ceti_column_meanings


,column,body_part,signal,axis_or_component,unit
0,Timestamp,global time,timestamp,time,ms
1,Pelvis_position_x,Pelvis,3D position,x,cm
2,Pelvis_position_y,Pelvis,3D position,y,cm
3,Pelvis_position_z,Pelvis,3D position,z,cm
4,Pelvis_extension,Pelvis,ORNT,NaN,deg
5,Pelvis_lateral_flexion_rotation,Pelvis,ORNT,NaN,deg
6,Pelvis_axial_rotation,Pelvis,ORNT,NaN,deg
7,LeftUpperLeg_position_x,LeftUpperLeg,3D position,x,cm
8,LeftUpperLeg_position_y,LeftUpperLeg,3D position,y,cm
9,LeftUpperLeg_position_z,LeftUpperLeg,3D position,z,cm


## 3. MobilePoser: tensor keys, dimensions, sensors, SMPL joints

这里不是 CSV column，而是 PyTorch `.pt` 里的 key 和 tensor 维度。下面会读取一个样例文件并输出每个 key 的 shape。


In [7]:
mobileposer_file = DATA_ROOT / 'mobileposer/TotalCapture.pt'
mobileposer = torch.load(mobileposer_file, map_location='cpu')

mobileposer_key_rows = []
for key, value in mobileposer.items():
    if isinstance(value, list) and value:
        mobileposer_key_rows.append({
            'key': key,
            'python_type': 'list',
            'num_sequences': len(value),
            'first_sequence_shape': tuple(value[0].shape) if hasattr(value[0], 'shape') else type(value[0]).__name__,
            'meaning': {
                'joint': 'SMPL 24 joint 3D positions, shape [T, 24, 3], treated as meters',
                'pose': 'SMPL 24 joint rotations, shape [T, 24, 3, 3]',
                'shape': 'SMPL beta/body-shape vector, shape [10]',
                'tran': 'global/root translation, shape [T, 3], treated as meters',
                'acc': '6 sparse IMU accelerations, shape [T, 6, 3]',
                'ori': '6 sparse IMU orientations, shape [T, 6, 3, 3]',
                'contact': 'foot/contact labels, shape [T, 2] when present',
            }.get(key, 'unknown')
        })
    else:
        mobileposer_key_rows.append({'key': key, 'python_type': type(value).__name__, 'num_sequences': None, 'first_sequence_shape': None, 'meaning': 'unknown'})

mobileposer_keys_df = pd.DataFrame(mobileposer_key_rows)
mobileposer_keys_df


,key,python_type,num_sequences,first_sequence_shape,meaning
0,joint,list,37,"(2057, 24, 3)","SMPL 24 joint 3D positions, shape [T, 24, 3], ..."
1,pose,list,37,"(2057, 24, 3, 3)","SMPL 24 joint rotations, shape [T, 24, 3, 3]"
2,shape,list,37,"(10,)","SMPL beta/body-shape vector, shape [10]"
3,tran,list,37,"(2057, 3)","global/root translation, shape [T, 3], treated..."
4,acc,list,37,"(2057, 6, 3)","6 sparse IMU accelerations, shape [T, 6, 3]"
5,ori,list,37,"(2057, 6, 3, 3)","6 sparse IMU orientations, shape [T, 6, 3, 3]"
6,contact,list,37,"(2057, 2)","foot/contact labels, shape [T, 2] when present"


In [8]:
mobileposer_imu_map = pd.DataFrame([
    {'sensor_index': 0, 'body_part': 'Pelvis', 'signals': 'ori[*,0] rotation matrix; acc[*,0] acceleration'},
    {'sensor_index': 1, 'body_part': 'LeftLowerLeg', 'signals': 'left shank IMU orientation + acceleration'},
    {'sensor_index': 2, 'body_part': 'RightLowerLeg', 'signals': 'right shank IMU orientation + acceleration'},
    {'sensor_index': 3, 'body_part': 'Head', 'signals': 'head IMU orientation + acceleration'},
    {'sensor_index': 4, 'body_part': 'LeftForeArm', 'signals': 'left forearm IMU orientation + acceleration'},
    {'sensor_index': 5, 'body_part': 'RightForeArm', 'signals': 'right forearm IMU orientation + acceleration'},
])

smpl_24 = [
    'Pelvis', 'LeftHip', 'RightHip', 'Spine1', 'LeftKnee', 'RightKnee', 'Spine2', 'LeftAnkle', 'RightAnkle', 'Spine3',
    'LeftFoot', 'RightFoot', 'Neck', 'LeftCollar', 'RightCollar', 'Head', 'LeftShoulder', 'RightShoulder',
    'LeftElbow', 'RightElbow', 'LeftWrist', 'RightWrist', 'LeftHand', 'RightHand'
]
smpl_joint_map = pd.DataFrame({'joint_index': range(len(smpl_24)), 'body_part': smpl_24, 'signals': 'joint position and pose rotation'})

print('MobilePoser IMU sensor mapping')
display(mobileposer_imu_map)
print('SMPL 24 joint order used by joint/pose')
display(smpl_joint_map)


MobilePoser IMU sensor mapping


,sensor_index,body_part,signals
0,0,Pelvis,"ori[*,0] rotation matrix; acc[*,0] acceleration"
1,1,LeftLowerLeg,left shank IMU orientation + acceleration
2,2,RightLowerLeg,right shank IMU orientation + acceleration
3,3,Head,head IMU orientation + acceleration
4,4,LeftForeArm,left forearm IMU orientation + acceleration
5,5,RightForeArm,right forearm IMU orientation + acceleration


SMPL 24 joint order used by joint/pose


,joint_index,body_part,signals
0,0,Pelvis,joint position and pose rotation
1,1,LeftHip,joint position and pose rotation
2,2,RightHip,joint position and pose rotation
3,3,Spine1,joint position and pose rotation
4,4,LeftKnee,joint position and pose rotation
5,5,RightKnee,joint position and pose rotation
6,6,Spine2,joint position and pose rotation
7,7,LeftAnkle,joint position and pose rotation
8,8,RightAnkle,joint position and pose rotation
9,9,Spine3,joint position and pose rotation


## 4. VIDIMU: CSV keypoint columns and IMU quaternion files

CSV 的每三列是一个 keypoint 的 `x/y/z`。STO header 里写着 `DataType=Quaternion`，每个 IMU column 是一个 quaternion。


In [9]:
vidimu_csv = DATA_ROOT / 'vidimu/dataset/videoandimus/S40/S40_A01_T01.csv'
vidimu_npose_csv = DATA_ROOT / 'vidimu/dataset/videoandimus/S40/S40_A01_T01_Npose.csv'
vidimu_sto = DATA_ROOT / 'vidimu/dataset/videoandimus/S40/S40_A01_T01.sto'

vidimu_header = pd.read_csv(vidimu_csv, nrows=0).columns.tolist()
vidimu_rows = []
for col in vidimu_header:
    m = re.match(r'(.+)_([xyz])$', col.strip())
    if m:
        vidimu_rows.append({'column': col, 'body_part': m.group(1), 'signal': '3D video/IK keypoint coordinate', 'axis': m.group(2), 'unit_assumption': 'mm-scale, converted to meters by /1000 in current converter'})
    else:
        vidimu_rows.append({'column': col, 'body_part': '', 'signal': 'unknown', 'axis': '', 'unit_assumption': ''})
vidimu_column_meanings = pd.DataFrame(vidimu_rows)

sto_header_lines = []
with open(vidimu_sto) as f:
    for line in f:
        sto_header_lines.append(line.rstrip('\n'))
        if line.strip() == 'endheader':
            break
    sto_cols = next(f).strip().split('\t')

print('VIDIMU CSV:', vidimu_csv)
print('n CSV columns:', len(vidimu_header))
print('STO header:')
print('\n'.join(sto_header_lines))
print('STO columns:', sto_cols)
vidimu_column_meanings


VIDIMU CSV: /home/pengfei/Downloads/data/vidimu/dataset/videoandimus/S40/S40_A01_T01.csv
n CSV columns: 102
STO header:
DataRate=50
DataType=Quaternion
version=3
OpenSimVersion=4.3-2021-08-27-4bc7ad9
endheader
STO columns: ['time', 'pelvis_imu', 'femur_r_imu', 'tibia_r_imu', 'femur_l_imu', 'tibia_l_imu']


,column,body_part,signal,axis,unit_assumption
0,pelvis_x,pelvis,3D video/IK keypoint coordinate,x,"mm-scale, converted to meters by /1000 in curr..."
1,pelvis_y,pelvis,3D video/IK keypoint coordinate,y,"mm-scale, converted to meters by /1000 in curr..."
2,pelvis_z,pelvis,3D video/IK keypoint coordinate,z,"mm-scale, converted to meters by /1000 in curr..."
3,left_hip_x,left_hip,3D video/IK keypoint coordinate,x,"mm-scale, converted to meters by /1000 in curr..."
4,left_hip_y,left_hip,3D video/IK keypoint coordinate,y,"mm-scale, converted to meters by /1000 in curr..."
5,left_hip_z,left_hip,3D video/IK keypoint coordinate,z,"mm-scale, converted to meters by /1000 in curr..."
6,right_hip_x,right_hip,3D video/IK keypoint coordinate,x,"mm-scale, converted to meters by /1000 in curr..."
7,right_hip_y,right_hip,3D video/IK keypoint coordinate,y,"mm-scale, converted to meters by /1000 in curr..."
8,right_hip_z,right_hip,3D video/IK keypoint coordinate,z,"mm-scale, converted to meters by /1000 in curr..."
9,torso_x,torso,3D video/IK keypoint coordinate,x,"mm-scale, converted to meters by /1000 in curr..."


In [10]:
vidimu_imu_map = pd.DataFrame([
    {'sto_column': 'pelvis_imu', 'body_part': 'Pelvis', 'signal': 'quaternion orientation', 'unit_or_type': 'unit quaternion, 50 Hz'},
    {'sto_column': 'femur_r_imu', 'body_part': 'RightUpperLeg / right femur', 'signal': 'quaternion orientation', 'unit_or_type': 'unit quaternion, 50 Hz'},
    {'sto_column': 'tibia_r_imu', 'body_part': 'RightLowerLeg / right tibia', 'signal': 'quaternion orientation', 'unit_or_type': 'unit quaternion, 50 Hz'},
    {'sto_column': 'femur_l_imu', 'body_part': 'LeftUpperLeg / left femur', 'signal': 'quaternion orientation', 'unit_or_type': 'unit quaternion, 50 Hz'},
    {'sto_column': 'tibia_l_imu', 'body_part': 'LeftLowerLeg / left tibia', 'signal': 'quaternion orientation', 'unit_or_type': 'unit quaternion, 50 Hz'},
])
vidimu_imu_map


,sto_column,body_part,signal,unit_or_type
0,pelvis_imu,Pelvis,quaternion orientation,"unit quaternion, 50 Hz"
1,femur_r_imu,RightUpperLeg / right femur,quaternion orientation,"unit quaternion, 50 Hz"
2,tibia_r_imu,RightLowerLeg / right tibia,quaternion orientation,"unit quaternion, 50 Hz"
3,femur_l_imu,LeftUpperLeg / left femur,quaternion orientation,"unit quaternion, 50 Hz"
4,tibia_l_imu,LeftLowerLeg / left tibia,quaternion orientation,"unit quaternion, 50 Hz"


## 5. ULTRA-MoCap: IMU/EMG/IK/TRC columns and body parts

ULTRA 同时有几种文件：`IMU/*.csv`、`EMG/*.csv`、`Processed IK/*.csv`、`Raw Marker/*.trc`。下面分别抽样输出 header 和解释。


In [11]:
ultra_imu_csv = DATA_ROOT / 'ULTRA-MoCap/ULTra-MoCap-raw-0/P01/Arm Swing/IMU/P01_ArmSwing_Fast.csv'
ultra_emg_csv = DATA_ROOT / 'ULTRA-MoCap/ULTra-MoCap-raw-0/P01/Arm Swing/EMG/P01_ArmSwing_Fast.csv'
ultra_ik_csv = DATA_ROOT / 'ULTRA-MoCap/ULTra-MoCap-raw-0/P01/Arm Swing/Processed IK/P01_ArmSwing_Fast.csv'
ultra_trc = DATA_ROOT / 'ULTRA-MoCap/ULTra-MoCap-raw-0/P01/Arm Swing/Raw Marker/P01_ArmSwing_Fast.trc'

def csv_header(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

ultra_headers = {
    'imu_csv': csv_header(ultra_imu_csv),
    'emg_csv': csv_header(ultra_emg_csv),
    'processed_ik_csv': csv_header(ultra_ik_csv),
}
{k: (len(v), v[:80]) for k, v in ultra_headers.items()}


{'imu_csv': (38,
  ['Frame',
   'Sub Frame',
   'ACCX1',
   'ACCY1',
   'ACCZ1',
   'GYROX1',
   'GYROY1',
   'GYROZ1',
   'ACCX2',
   'ACCY2',
   'ACCZ2',
   'GYROX2',
   'GYROY2',
   'GYROZ2',
   'ACCX3',
   'ACCY3',
   'ACCZ3',
   'GYROX3',
   'GYROY3',
   'GYROZ3',
   'ACCX4',
   'ACCY4',
   'ACCZ4',
   'GYROX4',
   'GYROY4',
   'GYROZ4',
   'ACCX5',
   'ACCY5',
   'ACCZ5',
   'GYROX5',
   'GYROY5',
   'GYROZ5',
   'ACCX6',
   'ACCY6',
   'ACCZ6',
   'GYROX6',
   'GYROY6',
   'GYROZ6']),
 'emg_csv': (5, ['Frame', 'Sub Frame', 'IM EMG4', 'IM EMG5', 'IM EMG6']),
 'processed_ik_csv': (55,
  ['time',
   'pelvis_tilt',
   'pelvis_list',
   'pelvis_rotation',
   'pelvis_tx',
   'pelvis_ty',
   'pelvis_tz',
   'hip_flexion_r',
   'hip_adduction_r',
   'hip_rotation_r',
   'hip_flexion_l',
   'hip_adduction_l',
   'hip_rotation_l',
   'flex_extension',
   'lat_bending',
   'axial_rotation',
   'L4_L5_IVDjnt_r3',
   'L4_L5_IVD_bendjnt_r1',
   'L4_L5_IVD_rotjnt_r2',
   'L3_L4_IVDjnt_r3',
   

In [12]:
# IMU sensor body-part mapping used by our current converter. This needs checking against ULTRA documentation.
ultra_imu_sensor_map = pd.DataFrame([
    {'sensor_id_n': 1, 'columns': 'ACCX1/ACCY1/ACCZ1, GYROX1/GYROY1/GYROZ1', 'mapped_body_part_in_current_code': 'Pelvis', 'status': 'converter assumption; verify with ULTRA docs'},
    {'sensor_id_n': 2, 'columns': 'ACCX2/ACCY2/ACCZ2, GYROX2/GYROY2/GYROZ2', 'mapped_body_part_in_current_code': 'T8 / upper trunk', 'status': 'converter assumption; verify with ULTRA docs'},
    {'sensor_id_n': 3, 'columns': 'ACCX3/ACCY3/ACCZ3, GYROX3/GYROY3/GYROZ3', 'mapped_body_part_in_current_code': 'LeftUpperArm', 'status': 'converter assumption; verify with ULTRA docs'},
    {'sensor_id_n': 4, 'columns': 'ACCX4/ACCY4/ACCZ4, GYROX4/GYROY4/GYROZ4', 'mapped_body_part_in_current_code': 'RightUpperArm', 'status': 'converter assumption; verify with ULTRA docs'},
    {'sensor_id_n': 5, 'columns': 'ACCX5/ACCY5/ACCZ5, GYROX5/GYROY5/GYROZ5', 'mapped_body_part_in_current_code': 'LeftForeArm', 'status': 'converter assumption; verify with ULTRA docs'},
    {'sensor_id_n': 6, 'columns': 'ACCX6/ACCY6/ACCZ6, GYROX6/GYROY6/GYROZ6', 'mapped_body_part_in_current_code': 'RightForeArm', 'status': 'converter assumption; verify with ULTRA docs'},
])

ik_col_meanings = []
for col in ultra_headers['processed_ik_csv']:
    if col == 'time':
        ik_col_meanings.append({'column': col, 'body_part_or_joint': 'time', 'signal': 'time', 'unit_or_type': 'seconds likely'})
    elif col.endswith(('_tx','_ty','_tz')):
        ik_col_meanings.append({'column': col, 'body_part_or_joint': col.rsplit('_', 1)[0], 'signal': 'translation coordinate', 'unit_or_type': 'OpenSim IK translation, likely meters'})
    else:
        ik_col_meanings.append({'column': col, 'body_part_or_joint': col, 'signal': 'joint angle/coordinate', 'unit_or_type': 'OpenSim IK coordinate, commonly degrees for rotations'})
ik_col_meanings = pd.DataFrame(ik_col_meanings)

print('ULTRA IMU sensor mapping used in current converter')
display(ultra_imu_sensor_map)
print('ULTRA Processed IK columns')
display(ik_col_meanings)


ULTRA IMU sensor mapping used in current converter


,sensor_id_n,columns,mapped_body_part_in_current_code,status
0,1,"ACCX1/ACCY1/ACCZ1, GYROX1/GYROY1/GYROZ1",Pelvis,converter assumption; verify with ULTRA docs
1,2,"ACCX2/ACCY2/ACCZ2, GYROX2/GYROY2/GYROZ2",T8 / upper trunk,converter assumption; verify with ULTRA docs
2,3,"ACCX3/ACCY3/ACCZ3, GYROX3/GYROY3/GYROZ3",LeftUpperArm,converter assumption; verify with ULTRA docs
3,4,"ACCX4/ACCY4/ACCZ4, GYROX4/GYROY4/GYROZ4",RightUpperArm,converter assumption; verify with ULTRA docs
4,5,"ACCX5/ACCY5/ACCZ5, GYROX5/GYROY5/GYROZ5",LeftForeArm,converter assumption; verify with ULTRA docs
5,6,"ACCX6/ACCY6/ACCZ6, GYROX6/GYROY6/GYROZ6",RightForeArm,converter assumption; verify with ULTRA docs


ULTRA Processed IK columns


,column,body_part_or_joint,signal,unit_or_type
0,time,time,time,seconds likely
1,pelvis_tilt,pelvis_tilt,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."
2,pelvis_list,pelvis_list,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."
3,pelvis_rotation,pelvis_rotation,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."
4,pelvis_tx,pelvis,translation coordinate,"OpenSim IK translation, likely meters"
5,pelvis_ty,pelvis,translation coordinate,"OpenSim IK translation, likely meters"
6,pelvis_tz,pelvis,translation coordinate,"OpenSim IK translation, likely meters"
7,hip_flexion_r,hip_flexion_r,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."
8,hip_adduction_r,hip_adduction_r,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."
9,hip_rotation_r,hip_rotation_r,joint angle/coordinate,"OpenSim IK coordinate, commonly degrees for ro..."


In [13]:
def parse_trc_header(path):
    with open(path) as f:
        lines = [next(f).rstrip('\n') for _ in range(6)]
    rates = lines[2].split('\t')
    marker_names = [x for x in lines[3].split('\t')[2:] if x]
    return {
        'data_rate': rates[0] if rates else None,
        'camera_rate': rates[1] if len(rates) > 1 else None,
        'num_frames': rates[2] if len(rates) > 2 else None,
        'num_markers': rates[3] if len(rates) > 3 else None,
        'units': rates[4] if len(rates) > 4 else None,
        'marker_names': marker_names,
    }

ULTRA_MARKER_ALIASES = {
    'LASI': 'Pelvis', 'RASI': 'Pelvis', 'LPSI': 'Pelvis', 'RPSI': 'Pelvis',
    'LLFC': 'LeftUpperLeg', 'RLFC': 'RightUpperLeg',
    'LKNE': 'LeftLowerLeg', 'RKNE': 'RightLowerLeg',
    'LLAN': 'LeftFoot', 'RLAN': 'RightFoot', 'LHEE': 'LeftFoot', 'RHEE': 'RightFoot',
    'LTOE': 'LeftToe', 'RTOE': 'RightToe',
}

trc_info = parse_trc_header(ultra_trc)
trc_marker_df = pd.DataFrame([
    {'marker_name': m, 'mapped_body_part_in_current_converter': ULTRA_MARKER_ALIASES.get(m, ''), 'signal': '3D marker position x/y/z', 'unit': trc_info['units']}
    for m in trc_info['marker_names']
])
print('TRC info:', {k:v for k,v in trc_info.items() if k != 'marker_names'})
trc_marker_df


TRC info: {'data_rate': '100.0', 'camera_rate': '100.0', 'num_frames': '3229', 'num_markers': '60', 'units': 'mm'}


,marker_name,mapped_body_part_in_current_converter,signal,unit
0,HEAD,,3D marker position x/y/z,mm
1,C7,,3D marker position x/y/z,mm
2,CLAV,,3D marker position x/y/z,mm
3,STUR,,3D marker position x/y/z,mm
4,LSHO,,3D marker position x/y/z,mm
5,LHAR2,,3D marker position x/y/z,mm
6,LHME,,3D marker position x/y/z,mm
7,LRAH,,3D marker position x/y/z,mm
8,LRAD,,3D marker position x/y/z,mm
9,LULN,,3D marker position x/y/z,mm


## 6. KeraalDataset: Vicon/Kinect/OpenPose/BlazePose body-part order

这里的顺序来自本地 `data/KeraalDataset/archives/readme_files_format.txt`。Vicon/Kinect 不是带 header 的 CSV，而是每行按固定 body-part 顺序拼接 `x,y,z,qx,qy,qz,qw`。


In [14]:
keraal_vicon_targets = [
    'Right Forearm', 'Left Forearm', 'Right Arm', 'Left Arm', 'Chest', 'Right Thigh', 'Left Thigh',
    'Right Shoulder', 'Left Shoulder', 'Right Hand', 'Left Hand', 'Right Foot', 'Left Foot', 'Hips',
    'Head', 'Right Tibia', 'Left Tibia'
]
keraal_kinect_joints = [
    'SpineBase', 'SpineMid', 'Neck', 'Head', 'ShoulderLeft', 'ElbowLeft', 'WristLeft', 'HandLeft',
    'ShoulderRight', 'ElbowRight', 'WristRight', 'HandRight', 'HipLeft', 'KneeLeft', 'AnkleLeft', 'FootLeft',
    'HipRight', 'KneeRight', 'AnkleRight', 'FootRight', 'SpineShoulder', 'HandTipLeft', 'ThumbLeft',
    'HandTipRight', 'ThumbRight'
]
keraal_openpose_joints = [
    'Head', 'mShoulder', 'rShoulder', 'rElbow', 'rWrist', 'lShoulder', 'lElbow', 'lWrist',
    'rHip', 'rKnee', 'rAnkle', 'lHip', 'lKnee', 'lAnkle'
]
keraal_blazepose_joints = [
    'Nose', 'Left_eye_inner', 'Left_eye', 'Left_eye_outer', 'Right_eye_inner', 'Right_eye', 'Right_eye_outer',
    'Left_ear', 'Right_ear', 'Mouth_left', 'Mouth_right', 'Left_shoulder', 'Right_shoulder', 'Left_elbow',
    'Right_elbow', 'Left_wrist', 'Right_wrist', 'Left_pinky', 'Right_pinky', 'Left_index', 'Right_index',
    'Left_thumb', 'Right_thumb', 'Left_hip', 'Right_hip', 'Left_knee', 'Right_knee', 'Left_ankle', 'Right_ankle',
    'Left_heel', 'Right_heel', 'Left_foot_index', 'Right_foot_index'
]

keraal_tables = {
    'vicon_targets_17': pd.DataFrame({'index': range(17), 'body_part': keraal_vicon_targets, 'per_body_part_values': 'x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w_quat'}),
    'kinect_joints_25': pd.DataFrame({'index': range(25), 'body_part': keraal_kinect_joints, 'per_body_part_values': 'x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w_quat'}),
    'openpose_coco_14': pd.DataFrame({'index': range(14), 'body_part': keraal_openpose_joints, 'per_body_part_values': 'x_pos, y_pos'}),
    'blazepose_33': pd.DataFrame({'index': range(33), 'body_part': keraal_blazepose_joints, 'per_body_part_values': 'x_pos, y_pos, z_pos'}),
}
for name, df in keraal_tables.items():
    print('\n', name)
    display(df)



 vicon_targets_17


,index,body_part,per_body_part_values
0,0,Right Forearm,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
1,1,Left Forearm,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
2,2,Right Arm,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
3,3,Left Arm,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
4,4,Chest,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
5,5,Right Thigh,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
6,6,Left Thigh,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
7,7,Right Shoulder,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
8,8,Left Shoulder,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
9,9,Right Hand,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."



 kinect_joints_25


,index,body_part,per_body_part_values
0,0,SpineBase,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
1,1,SpineMid,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
2,2,Neck,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
3,3,Head,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
4,4,ShoulderLeft,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
5,5,ElbowLeft,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
6,6,WristLeft,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
7,7,HandLeft,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
8,8,ShoulderRight,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."
9,9,ElbowRight,"x_pos, y_pos, z_pos, x_quat, y_quat, z_quat, w..."



 openpose_coco_14


,index,body_part,per_body_part_values
0,0,Head,"x_pos, y_pos"
1,1,mShoulder,"x_pos, y_pos"
2,2,rShoulder,"x_pos, y_pos"
3,3,rElbow,"x_pos, y_pos"
4,4,rWrist,"x_pos, y_pos"
5,5,lShoulder,"x_pos, y_pos"
6,6,lElbow,"x_pos, y_pos"
7,7,lWrist,"x_pos, y_pos"
8,8,rHip,"x_pos, y_pos"
9,9,rKnee,"x_pos, y_pos"



 blazepose_33


,index,body_part,per_body_part_values
0,0,Nose,"x_pos, y_pos, z_pos"
1,1,Left_eye_inner,"x_pos, y_pos, z_pos"
2,2,Left_eye,"x_pos, y_pos, z_pos"
3,3,Left_eye_outer,"x_pos, y_pos, z_pos"
4,4,Right_eye_inner,"x_pos, y_pos, z_pos"
5,5,Right_eye,"x_pos, y_pos, z_pos"
6,6,Right_eye_outer,"x_pos, y_pos, z_pos"
7,7,Left_ear,"x_pos, y_pos, z_pos"
8,8,Right_ear,"x_pos, y_pos, z_pos"
9,9,Mouth_left,"x_pos, y_pos, z_pos"


## 7. rehab_data / 12112951: EMG/IMU sensor placement and labels

这些信息来自本地 `metadata.txt`。它是 rehab 动作分类/传感器数据，不是 full-body joint GT。


In [15]:
rehab_sensor_placement = pd.DataFrame([
    {'sensor_id': 1, 'body_side': 'right', 'muscle_or_location': 'rectus femoris', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 2, 'body_side': 'right', 'muscle_or_location': 'hamstrings', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 3, 'body_side': 'right', 'muscle_or_location': 'tibialis anterior', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 4, 'body_side': 'right', 'muscle_or_location': 'gastrocnemius', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 5, 'body_side': 'left', 'muscle_or_location': 'rectus femoris', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 6, 'body_side': 'left', 'muscle_or_location': 'hamstrings', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 7, 'body_side': 'left', 'muscle_or_location': 'tibialis anterior', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
    {'sensor_id': 8, 'body_side': 'left', 'muscle_or_location': 'gastrocnemius', 'signals': 'EMG, acc x/y/z, gyro x/y/z'},
])
rehab_units = pd.DataFrame([
    {'signal': 'EMG', 'unit': 'millivolts', 'sampling_rate_hz': 1259.2592592592594},
    {'signal': 'accelerometer x/y/z', 'unit': 'G', 'sampling_rate_hz': 148.14814814814815},
    {'signal': 'gyroscope x/y/z', 'unit': 'deg/s', 'sampling_rate_hz': 148.14814814814815},
])
rehab_labels = pd.DataFrame([
    {'label_id': 0, 'execution': 'Correct', 'details': 'Squat'},
    {'label_id': 1, 'execution': 'Wrong', 'details': 'Squat with weight transfer on healthy leg'},
    {'label_id': 2, 'execution': 'Wrong', 'details': 'Squat placing injured leg in front'},
    {'label_id': 3, 'execution': 'Correct', 'details': 'Seated leg extension'},
    {'label_id': 4, 'execution': 'Wrong', 'details': 'Seated leg extension of injured knee without full range of motion'},
    {'label_id': 5, 'execution': 'Wrong', 'details': 'Seated leg extension of injured knee with lifting limb from chair'},
    {'label_id': 6, 'execution': 'Correct', 'details': 'Walking'},
    {'label_id': 7, 'execution': 'Wrong', 'details': 'Walking with injured knee not fully extended'},
    {'label_id': 8, 'execution': 'Wrong', 'details': 'Walking with injured limb in full knee extension with hip abduction'},
])
print('Sensor placement')
display(rehab_sensor_placement)
print('Units and frequency')
display(rehab_units)
print('Labels')
display(rehab_labels)


Sensor placement


,sensor_id,body_side,muscle_or_location,signals
0,1,right,rectus femoris,"EMG, acc x/y/z, gyro x/y/z"
1,2,right,hamstrings,"EMG, acc x/y/z, gyro x/y/z"
2,3,right,tibialis anterior,"EMG, acc x/y/z, gyro x/y/z"
3,4,right,gastrocnemius,"EMG, acc x/y/z, gyro x/y/z"
4,5,left,rectus femoris,"EMG, acc x/y/z, gyro x/y/z"
5,6,left,hamstrings,"EMG, acc x/y/z, gyro x/y/z"
6,7,left,tibialis anterior,"EMG, acc x/y/z, gyro x/y/z"
7,8,left,gastrocnemius,"EMG, acc x/y/z, gyro x/y/z"


Units and frequency


,signal,unit,sampling_rate_hz
0,EMG,millivolts,1259.259259
1,accelerometer x/y/z,G,148.148148
2,gyroscope x/y/z,deg/s,148.148148


Labels


,label_id,execution,details
0,0,Correct,Squat
1,1,Wrong,Squat with weight transfer on healthy leg
2,2,Wrong,Squat placing injured leg in front
3,3,Correct,Seated leg extension
4,4,Wrong,Seated leg extension of injured knee without f...
5,5,Wrong,Seated leg extension of injured knee with lift...
6,6,Correct,Walking
7,7,Wrong,Walking with injured knee not fully extended
8,8,Wrong,Walking with injured limb in full knee extensi...


## 8. Quick file-count sanity check

这个 cell 用来确认每个数据集有哪些主要文件类型和数量。它不读大文件内容，只数文件。


In [16]:
def count_files(root, patterns):
    root = Path(root)
    out = []
    for label, pattern in patterns:
        files = list(root.glob(pattern))
        out.append({'label': label, 'pattern': str(root / pattern), 'count': len(files), 'example': str(files[0]) if files else ''})
    return pd.DataFrame(out)

file_counts = pd.concat([
    count_files(DATA_ROOT / 'CeTI-Age-Kinematics/ceti-age-kinematics', [('motion_tsv', '**/*_motion.tsv'), ('channels_tsv', '**/*_channels.tsv'), ('json', '**/*.json')]).assign(dataset='CeTI-Age-Kinematics'),
    count_files(DATA_ROOT / 'mobileposer', [('pt', '**/*.pt')]).assign(dataset='MobilePoser'),
    count_files(DATA_ROOT / 'vidimu', [('csv', '**/*.csv'), ('raw', '**/*.raw'), ('sto', '**/*.sto')]).assign(dataset='VIDIMU'),
    count_files(DATA_ROOT / 'ULTRA-MoCap', [('imu_csv', '**/IMU/*.csv'), ('emg_csv', '**/EMG/*.csv'), ('ik_csv', '**/Processed IK/*.csv'), ('trc', '**/*.trc')]).assign(dataset='ULTRA-MoCap'),
    count_files(DATA_ROOT / 'KeraalDataset', [('vicon', '**/vicon/**/*'), ('kinect', '**/kinect/**/*'), ('openpose_json', '**/openpose/**/*.json'), ('blazepose_json', '**/blazepose/**/*.json'), ('annotations', '**/*.anvil')]).assign(dataset='KeraalDataset'),
    count_files(DATA_ROOT / 'rehab_data', [('zip', '**/*.zip'), ('metadata', '**/metadata.txt')]).assign(dataset='rehab_data'),
], ignore_index=True)
file_counts[['dataset', 'label', 'count', 'example']]


,dataset,label,count,example
0,CeTI-Age-Kinematics,motion_tsv,1563,/home/pengfei/Downloads/data/CeTI-Age-Kinemati...
1,CeTI-Age-Kinematics,channels_tsv,1563,/home/pengfei/Downloads/data/CeTI-Age-Kinemati...
2,CeTI-Age-Kinematics,json,1565,/home/pengfei/Downloads/data/CeTI-Age-Kinemati...
3,MobilePoser,pt,23,/home/pengfei/Downloads/data/mobileposer/BMLmo...
4,VIDIMU,csv,1091,/home/pengfei/Downloads/data/vidimu/dataset/vi...
5,VIDIMU,raw,242,/home/pengfei/Downloads/data/vidimu/dataset/vi...
6,VIDIMU,sto,416,/home/pengfei/Downloads/data/vidimu/dataset/vi...
7,ULTRA-MoCap,imu_csv,217,/home/pengfei/Downloads/data/ULTRA-MoCap/ULTra...
8,ULTRA-MoCap,emg_csv,217,/home/pengfei/Downloads/data/ULTRA-MoCap/ULTra...
9,ULTRA-MoCap,ik_csv,208,/home/pengfei/Downloads/data/ULTRA-MoCap/ULTra...
